# 03 ? Jalankan pipeline dan simpan satu run lengkap

Menjalankan sel pipeline akan membuat folder run baru. Parameter masalah tetap di `config/v2.yaml`, sedangkan lokasi penyimpanan dan nama submission diatur oleh `config/artifacts.yaml`.

Pipeline menyimpan fitur, semua return core, ambiguitas, hasil video jika tersedia, refinement, diagnostik, submission, dan validasi sebagai CSV/JSON. Core awal dan core setelah video disimpan terpisah. Run gagal memiliki manifest `failed`; run selesai memiliki manifest `completed`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'games.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.artifacts import list_runs, read_json, read_table


## Dua konfigurasi terpisah

In [10]:
import yaml
from src.pipeline import run_pipeline
MODEL_CONFIG = ROOT / 'config/v3.yaml'
STORAGE_CONFIG = ROOT / 'config/artifacts.yaml'
model_config = yaml.safe_load(MODEL_CONFIG.read_text(encoding='utf-8-sig'))
display(model_config)
display(yaml.safe_load(STORAGE_CONFIG.read_text(encoding='utf-8-sig')))


{'version': 3,
 'game': 'BAR-ATH',
 'source': 'tactical',
 'fps': 25,
 'pitch': [105.3, 68.0],
 'quality': {'max_ball_speed_mps': 45.0},
 'events': {'anchor_window_frames': 25},
 'control': {'distance_scale_m': 2.5,
  'distance_weight': 1.0,
  'tag_weight': 0.3,
  'persistence_weight': 0.5,
  'margin_weight': 0.25,
  'proximity_window_frames': 11,
  'transition_window_frames': 9},
 'live_dead': {'live_prior_score': 0.25,
  'dead_prior_score': 0.25,
  'ball_moving_live_weight': 0.22,
  'player_motion_live_weight': 0.12,
  'out_of_pitch_weight': 0.35,
  'stoppage_anchor_weight': 0.32,
  'restart_anchor_weight': 0.14,
  'stationary_near_anchor_weight': 0.12},
 'fusion': {'team_weight': 1.0,
  'live_weight': 1.0,
  'dead_weight': 1.0,
  'dead_bias': 0.0,
  'second_half_kickoff_frame': 113742},
 'temporal': {'ha_cost': 2.4,
  'hd_cost': 1.65,
  'dh_cost': 1.9,
  'turnover_discount': 0.8,
  'stoppage_discount': 0.7,
  'restart_discount': 0.7,
  'minimum_transition_cost': 0.05,
  'short_spell

{'root_dir': 'artifacts', 'submission_filename': 'baseline_v2.csv'}

## Eksekusi

`RUN_PIPELINE=True` menjalankan pipeline sebenarnya. Gunakan `False` untuk membuka hasil terakhir tanpa menjalankan ulang. Tidak ada logika prediksi atau diagnostik tambahan di cell.

In [11]:
RUN_PIPELINE = True
if RUN_PIPELINE:
    result = run_pipeline(model_config, project_dir=ROOT, storage_config_path=STORAGE_CONFIG,
                          execution_metadata={'notebook': 'notebooks/03_baseline_possession.ipynb',
                                              'model_config_source': str(MODEL_CONFIG)})
    RUN_DIR = Path(result.artifact_dir)
else:
    runs = [r for r in list_runs(project_dir=ROOT) if r['entrypoint'] == 'run_pipeline']
    if not runs:
        raise RuntimeError('Belum ada run selesai.')
    RUN_DIR = Path(runs[-1]['path'])
manifest = read_json(RUN_DIR)
print('Run:', manifest['run_id'])
print('Status:', manifest['status'])
print('Folder:', RUN_DIR)


Run: 20261115_213258_734998_6f1c5833
Status: completed
Folder: C:\Users\HP 840\Documents\Hizkia ITS\Lomba\MBG\more-than-a-hack-mbg\artifacts\20261115_213258_734998_6f1c5833


## Hasil yang disimpan

In [9]:
display(read_json(RUN_DIR, 'config/model.json'))
display(read_json(RUN_DIR, 'config/storage.json'))
display(read_json(RUN_DIR, 'submission/write/result.json'))
display(read_json(RUN_DIR, 'submission/validator/result.json'))
display(read_json(RUN_DIR, 'submission/diagnostics.json'))


{'version': 2,
 'game': 'BAR-ATH',
 'source': 'tactical',
 'fps': 25,
 'pitch': [105.3, 68.0],
 'quality': {'max_ball_speed_mps': 45.0},
 'events': {'anchor_window_frames': 25},
 'control': {'distance_scale_m': 2.5,
  'distance_weight': 1.0,
  'tag_weight': 0.3,
  'persistence_weight': 0.5,
  'margin_weight': 0.25,
  'proximity_window_frames': 11,
  'transition_window_frames': 9},
 'live_dead': {'live_prior_score': 0.55,
  'dead_prior_score': 0.25,
  'ball_moving_live_weight': 0.22,
  'player_motion_live_weight': 0.12,
  'out_of_pitch_weight': 0.35,
  'stoppage_anchor_weight': 0.32,
  'restart_anchor_weight': 0.14,
  'stationary_near_anchor_weight': 0.12},
 'fusion': {'team_weight': 1.0,
  'live_weight': 1.0,
  'dead_weight': 1.0,
  'dead_bias': 0.0,
  'second_half_kickoff_frame': 113742},
 'temporal': {'ha_cost': 2.4,
  'hd_cost': 1.65,
  'dh_cost': 1.9,
  'turnover_discount': 0.8,
  'stoppage_discount': 0.7,
  'restart_discount': 0.7,
  'minimum_transition_cost': 0.05,
  'short_spell

{'root_dir': 'artifacts',
 'submission_filename': 'baseline_v2.csv',
 'resolved_root_dir': 'C:\\Users\\HP 840\\Documents\\Hizkia ITS\\Lomba\\MBG\\more-than-a-hack-mbg\\artifacts',
 'source_file': 'C:\\Users\\HP 840\\Documents\\Hizkia ITS\\Lomba\\MBG\\more-than-a-hack-mbg\\config\\artifacts.yaml'}

{'path': 'C:\\Users\\HP 840\\Documents\\Hizkia ITS\\Lomba\\MBG\\more-than-a-hack-mbg\\artifacts\\20261115_152745_650609_05d95801\\submission\\baseline_v2.csv',
 'rows': 199319,
 'bytes': 1682773,
 'sha256': '04b4ec54fa9ceb03d92f983674e3e832c633916d6b70fe9b28fbc098a25f9022'}

{'status': 'passed',
 'message': 'OK    baseline_v2.csv\n      199319 rows, frame 0..199318\n\n      label counts\n        H    96210   48.3%\n        A    59713   30.0%\n        D     2934    1.5%\n        X    40462   20.3%\n\n      reports 1070 possessions of mean 5.8 s\n      (counted with each X as a gap; the scorer bridges abstentions inside a real\n       possession, so its count can be lower than this one)\n\nValid. Note this checks the format only -- it says nothing about whether the labels are right.\nScore it against the public ground truth with run_evaluation.py.'}

{'graded_frames': 158857,
 'frame_counts': {'H': 96210, 'A': 59713, 'D': 2934},
 'possession_spells': 1070,
 'dead_segments': 1,
 'total_transitions': 1069,
 'possession_median_s': 2.04,
 'possession_mean_s': 5.828897196261682,
 'dead_median_s': 117.36,
 'preliminary_transitions': 1069,
 'soft_short_spells_merged': 0,
 'boundaries_examined': 1069,
 'boundaries_shifted': 115,
 'video_windows_selected': 40,
 'video': {'status': 'disabled', 'accepted': 0},
 'ball_observed_share_graded': 0.593074274347369,
 'submission_sha256': '04b4ec54fa9ceb03d92f983674e3e832c633916d6b70fe9b28fbc098a25f9022',
 'config_sha256': 'cd942c2311157666dc07951563b9c726391ebaeb8a0892e798aac62c2f50bf3e',
 'validator': {'status': 'passed',
  'message': 'OK    baseline_v2.csv\n      199319 rows, frame 0..199318\n\n      label counts\n        H    96210   48.3%\n        A    59713   30.0%\n        D     2934    1.5%\n        X    40462   20.3%\n\n      reports 1070 possessions of mean 5.8 s\n      (counted with each X

## Lanjutkan interpretasi

Buka notebook 01, 02, dan 04. Pilih `RUN_ID` yang dicetak di atas agar seluruh pembahasan merujuk run yang sama. `manifest.json` mencatat daftar CSV/JSON, schema, hash sumber, dan status setiap tahap. Artefak run bersifat lokal dan dikecualikan dari Git.


run_id = 20261115_152745_650609_05d95801

## Interpretasi — Baseline Pipeline Execution

### 1. Definisi dan istilah

- **Pipeline:** Rangkaian pemrosesan dari data input, pembentukan fitur, penghitungan evidence, inference, hingga pembuatan submission.
- **Baseline:** Versi awal sistem yang digunakan sebagai acuan untuk mengevaluasi perubahan model berikutnya. Baseline tidak harus berupa model sederhana.
- **Configuration:** Kumpulan parameter yang mengatur perilaku setiap modul, termasuk bobot evidence, biaya transisi, dan refinement.
- **Inference:** Proses menghasilkan prediksi state berdasarkan fitur dan aturan model yang tersedia.
- **Temporal decoding:** Proses menentukan urutan state dengan mempertimbangkan evidence dan kontinuitas waktu.
- **Boundary refinement:** Penyesuaian posisi waktu pergantian state setelah decoding.
- **Artifact:** File hasil atau diagnostik yang disimpan pipeline, seperti CSV, JSON, label, skor, dan transition report.
- **Run ID:** Identitas unik sebuah eksekusi pipeline, sehingga hasil dapat ditelusuri ke konfigurasi dan kode yang digunakan.
- **Manifest:** Metadata eksekusi yang menyimpan status modul, daftar file, schema, dan checksum.
- **Validator:** Pemeriksaan apakah submission memenuhi format resmi kompetisi. Validator tidak mengukur akurasi prediksi.
- **Checksum (SHA-256):** Sidik jari digital yang membantu memverifikasi apakah suatu file atau konfigurasi identik dengan versi sebelumnya.

### 2. Interpretasi konfigurasi

Pipeline menggunakan:

- Data tactical Barcelona–Athletic Club.
- FPS sebesar 25.
- Ukuran lapangan 105,3 × 68 meter.
- Modul quality, event evidence, team control, live/dead, fusion, temporal decoder, dan boundary refinement.
- Video recovery dalam keadaan nonaktif (`video.enabled = False`).

Parameter seperti `ha_cost`, `hd_cost`, dan `dh_cost` merepresentasikan biaya perubahan state pada decoder. Semakin tinggi biaya transisi, semakin besar keuntungan evidence yang umumnya diperlukan untuk mengubah state, dengan tetap memperhatikan mekanisme diskon transisi yang berlaku.

`short_spell_soft_penalty` menyediakan penalti terhadap segmen pendek dalam proses temporal inference.

Konfigurasi juga menetapkan `second_half_kickoff_frame = 113742`, yang digunakan untuk memaksa interval awal babak kedua menjadi state D.

### 3. Interpretasi hasil eksekusi

Pipeline berhasil dijalankan dengan status `completed` dan menghasilkan `baseline_v2.csv`.

File submission memiliki:

- 199.319 baris data.
- Indeks frame 0 sampai 199318.
- Kolom `frame` dan `label`.
- Label yang valid: H, A, D, dan X.
- Status validator: `passed`.

Distribusi label pada seluruh submission:

| Label | Jumlah frame | Persentase |
|---|---:|---:|
| H | 96.210 | 48,3% |
| A | 59.713 | 30,0% |
| D | 2.934 | 1,5% |
| X | 40.462 | 20,3% |

Persentase tersebut menggunakan seluruh 199.319 frame sebagai penyebut, termasuk frame di luar wilayah penilaian.

Seluruh 40.462 frame X berada di luar blok graded pada run ini. Dengan demikian, seluruh graded frames memperoleh prediksi H, A, atau D.

Output juga melaporkan 1.070 predicted possession spells dengan durasi rata-rata sekitar 5,83 detik.

**Penting:** Jumlah possession merupakan hasil algoritma, bukan jumlah possession sebenarnya dalam pertandingan.

### 4. Validitas dan keterbatasan

Status `passed` menunjukkan bahwa format submission memenuhi persyaratan resmi, tetapi tidak membuktikan bahwa label possession benar.

Demikian pula, status `completed` menunjukkan pipeline berhasil dieksekusi, bukan bahwa seluruh komponen model menghasilkan keputusan yang tepat.

Karena video recovery tidak diaktifkan, tidak ada evidence tambahan dari pemrosesan ulang video pada run ini. `final_core_pass` adalah `initial`.

Manifest dan checksum memungkinkan hasil ini dipakai sebagai referensi eksperimen berikutnya.

### 5. Kesimpulan

Pipeline baseline v2 sudah memenuhi persyaratan teknis submission dan menyimpan artefak yang cukup lengkap untuk diagnosis.

Tahap berikutnya bukan memperbaiki format output, melainkan mengevaluasi perilaku model, terutama kemampuan mendeteksi dead ball dan stabilitas pergantian possession.

Setiap eksperimen berikutnya sebaiknya menyimpan run ID, konfigurasi, dan diagnostik agar perubahan hasil dapat ditelusuri secara reproducible.
